# Reranking: Skor SBERT + Skor Efisiensi Bahan

Lanjutan dari `02_embedding_indexing.ipynb`.

Ide dasar: retrieval SBERT murni bagus untuk menangkap kemiripan semantik umum ("masakan telur"), tapi tidak menjamin kelengkapan bahan sesuai input user. Di sini kita:
1. Retrieve kandidat lebih banyak dari SBERT/ChromaDB (misal top 30), bukan langsung top 5
2. Hitung **skor efisiensi bahan** = seberapa cocok/lengkap bahan resep dengan bahan yang dimiliki user
3. Gabungkan skor SBERT (dinormalisasi) + skor efisiensi bahan jadi skor akhir
4. Urutkan ulang (rerank), ambil top-N final untuk ditampilkan ke user

In [1]:
import re
import html
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Model & Koneksi ke ChromaDB

Pastikan `chroma_db/` hasil dari notebook 02 ada di direktori yang sama.

In [2]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

print("Jumlah dokumen di collection:", collection.count())


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9948.23it/s]


Jumlah dokumen di collection: 5000


## 2. Parser Bahan untuk Input User

Sama seperti parser di `01_preprocessing.ipynb` (disalin ke sini supaya notebook ini bisa jalan standalone). Kalau nanti diintegrasikan ke aplikasi, sebaiknya fungsi ini dipindah ke satu modul `.py` bersama supaya tidak duplikat.

In [3]:
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"
    r"|\d+\s+\d+\s*/\s*\d+"
    r"|\d+\s*/\s*\d+"
    r"|\d+[.,]\d+"
    r"|\d+"
    r"|sesuai selera"
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)
# catatan: \b sengaja HANYA di belakang unit, bukan di depan -- supaya
# qty+unit yang nempel tanpa spasi (mis. '100gr') tetap kebaca, tapi unit
# tidak match sebagian kata di tengah nama bahan (mis. 'l' di 'lengkuas')
INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?:(?P<unit>{UNIT_PATTERN})\b)?\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw).strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)
    main, note = (raw.split(",", 1) + [""])[:2] if "," in raw else (raw, "")
    main = main.strip()
    m = INGREDIENT_REGEX.match(main)
    if m and m.group("name"):
        name = clean_name(m.group("name"))
    else:
        name = clean_name(main)
    return {"raw": raw, "name": name}


def parse_user_ingredients(text: str) -> list:
    """Input: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
    Note: user memisahkan tiap bahan dgn koma, TAPI parse_ingredient juga memakai
    koma untuk memisahkan catatan (mis. 'dikocok lepas'). Untuk input user yang
    berupa daftar bahan bersih (tanpa catatan cara masak), kita split dulu per
    kemungkinan bahan pakai heuristik: pisah di koma yang diikuti oleh angka/qty
    word atau di awal token baru. Untuk kasus sederhana (tanpa catatan), split
    koma biasa sudah cukup."""
    items = [t.strip() for t in text.split(",") if t.strip()]
    parsed = [parse_ingredient(item) for item in items]
    return [p["name"] for p in parsed if p["name"]]


# test
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
user_ingredients = parse_user_ingredients(user_input)
print(user_ingredients)

['telur', 'dada ayam', 'bawang putih']


## 3. Skor Efisiensi Bahan

Definisi yang dipakai di sini:

- **matched** = jumlah bahan resep yang cocok dengan salah satu bahan user (matching berbasis substring dua arah, bukan exact match, supaya `"telur"` tetap match dengan `"telur ayam"`)
- **coverage_recipe** = matched / total bahan resep -> makin tinggi, makin sedikit bahan tambahan yang perlu dibeli user (ini inti "efisiensi")
- **coverage_user** = matched / total bahan yang diketik user -> makin tinggi, makin banyak bahan user yang benar-benar terpakai (bukan resep yang random cuma nyenggol 1 bahan)
- **efficiency_score** = harmonic mean dari `coverage_recipe` dan `coverage_user` (mirip F1-score) -> menyeimbangkan dua sisi tadi, dan resep yang timpang di salah satu sisi (misal cocok bahan tapi resepnya beda arah) akan dihukum

> Ini rule-based/baseline. Kalau di evaluasi hasilnya kurang bagus, opsi lanjutannya: matching pakai embedding similarity antar nama bahan (bukan cuma substring), atau pakai stemming/lemmatization Bahasa Indonesia (mis. Sastrawi).

In [4]:
def ingredient_matches(user_name: str, recipe_name: str) -> bool:
    u = user_name.lower().strip()
    r = recipe_name.lower().strip()
    if not u or not r:
        return False
    return u in r or r in u


def compute_efficiency_score(user_ingredients: list, recipe_ingredients: list) -> dict:
    if not recipe_ingredients:
        return {
            "matched": 0,
            "coverage_recipe": 0.0,
            "coverage_user": 0.0,
            "efficiency_score": 0.0,
            "matched_items": [],
            "missing_items": [],
        }

    matched_recipe_items = []
    matched_user_items = set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)

    matched = len(matched_recipe_items)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]

    coverage_recipe = matched / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )

    if coverage_recipe + coverage_user == 0:
        efficiency_score = 0.0
    else:
        efficiency_score = (2 * coverage_recipe * coverage_user) / (
            coverage_recipe + coverage_user
        )

    return {
        "matched": matched,
        "coverage_recipe": coverage_recipe,
        "coverage_user": coverage_user,
        "efficiency_score": efficiency_score,
        "matched_items": matched_recipe_items,
        "missing_items": missing_items,
    }


# test cepat
recipe_ex = ["telur ayam", "dada ayam fillet", "bawang putih", "kecap manis", "garam"]
print(compute_efficiency_score(user_ingredients, recipe_ex))

{'matched': 3, 'coverage_recipe': 0.6, 'coverage_user': 1.0, 'efficiency_score': 0.7499999999999999, 'matched_items': ['telur ayam', 'dada ayam fillet', 'bawang putih'], 'missing_items': ['kecap manis', 'garam']}


## 4. Fungsi Reranking Utuh

Alur: retrieve `top_k_candidates` dari ChromaDB (lebih banyak dari yang akan ditampilkan) -> hitung efficiency score tiap kandidat -> gabung dengan sbert_score (dinormalisasi 0-1 dgn min-max di antara kandidat) -> urutkan berdasar `final_score` -> potong ke `top_n_final`.

In [5]:
def retrieve_candidates(user_text: str, top_k: int = 30) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        candidates.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "steps_text": meta["steps_text"],
                "sbert_score": 1
                - results["distances"][0][i],  # cosine distance -> similarity
            }
        )
    return candidates


def rerank(
    user_text: str,
    top_k_candidates: int = 30,
    top_n_final: int = 5,
    w_sbert: float = 0.5,
    w_efficiency: float = 0.5,
) -> list:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates(user_text, top_k=top_k_candidates)

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = (
            w_sbert * float(s_norm) + w_efficiency * eff["efficiency_score"]
        )

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], user_ingredients

## 5. Uji Coba & Bandingkan dengan Retrieval Murni

In [6]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

final_results, parsed_user_ingredients = rerank(
    user_input, top_k_candidates=30, top_n_final=5, w_sbert=0.5, w_efficiency=0.5
)

print("Bahan user (setelah parsing):", parsed_user_ingredients)
print("=" * 70)

for i, c in enumerate(final_results):
    print(f"[{i + 1}] {c['title']}")
    print(
        f"    final_score={c['final_score']:.4f}  "
        f"(sbert_norm={c['sbert_score_norm']:.3f}, efficiency={c['efficiency_score']:.3f})"
    )
    print(f"    bahan resep: {c['ingredient_names']}")
    print(f"    -> sudah dimiliki user: {c['matched_items']}")
    print(f"    -> perlu ditambah: {c['missing_items']}")
    print()

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
[1] Telur Dadar Daun Bawang
    final_score=0.6250  (sbert_norm=1.000, efficiency=0.250)
    bahan resep: ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica bubuk']
    -> sudah dimiliki user: ['telur ayam']
    -> perlu ditambah: ['daun bawang merah', 'cabai', 'garam', 'merica bubuk']

[2] Telur Dadar
    final_score=0.5026  (sbert_norm=0.561, efficiency=0.444)
    bahan resep: ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[3] Telur Dadar
    final_score=0.4880  (sbert_norm=0.576, efficiency=0.400)
    bahan resep: ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 'lada bubuk', 'garam']
    -> sudah dimiliki user: ['telur ayam', 'bawang putih bubuk']
    -> perlu ditambah: ['kor

## Catatan

- `w_sbert` dan `w_efficiency` adalah hyperparameter — coba beberapa kombinasi (mis. 0.7/0.3, 0.3/0.7) dan bandingkan hasilnya secara kualitatif, atau kalau punya ground truth/labeled data, bisa dievaluasi kuantitatif (mis. NDCG, precision@k).
- `top_k_candidates` (jumlah kandidat sebelum rerank) juga berpengaruh: makin besar, makin besar peluang resep yang "lengkap bahannya" tapi skor SBERT-nya biasa-biasa saja ikut terangkat -- tapi juga makin lambat.
- Field `missing_items` di setiap hasil ini persis yang kamu butuh untuk fitur *"apakah ada bahan pelengkap lainnya?"* -- itu daftar bahan yang belum dikonfirmasi user, bisa langsung dilempar ke prompt Llama3 di tahap berikutnya.

## 6. Hybrid Retrieval: SBERT + Keyword Fallback

**Masalah:** untuk query pendek (cuma 2-3 bahan), retrieval SBERT murni bisa saja tidak pernah memasukkan resep yang benar-benar match ke dalam top-K kandidat sama sekali -- walau resep itu ada di dataset. Ini karena embedding semantik cenderung mengelompokkan berdasarkan pola judul/topik dominan (mis. "telur dadar"), bukan menjamin kelengkapan bahan.

**Solusi:** gabungkan dua sumber kandidat sebelum reranking:
1. **SBERT candidates** -- hasil `collection.query()` seperti biasa (menangkap kemiripan semantik/topik)
2. **Keyword candidates** -- resep yang bahannya match langsung (substring) dengan input user, diambil dari seluruh koleksi (bukan cuma top-K SBERT) -- ini menjamin resep yang benar-benar relevan tidak akan terlewat gara-gara skor SBERT-nya kebetulan rendah

Kandidat dari keyword search yang skor SBERT-nya belum ada (karena tidak ikut ke-retrieve oleh ChromaDB) akan dihitung on-the-fly dengan encode ulang lalu cosine similarity terhadap query embedding, supaya tetap bisa dibandingkan adil dengan kandidat dari SBERT.

In [7]:
# cache seluruh koleksi di memori sekali saja (5000 dokumen, ringan)
_ALL_DOCS_CACHE = None


def get_all_documents():
    global _ALL_DOCS_CACHE
    if _ALL_DOCS_CACHE is None:
        raw = collection.get(include=["metadatas", "documents", "embeddings"])
        _ALL_DOCS_CACHE = list(
            zip(raw["ids"], raw["metadatas"], raw["documents"], raw["embeddings"])
        )
    return _ALL_DOCS_CACHE


def keyword_candidates(
    user_ingredients: list, exclude_ids: set, max_candidates: int = 20
) -> list:
    """Cari resep di SELURUH koleksi yang bahannya match keyword dgn input user,
    di luar id yang sudah ada di hasil SBERT."""
    scored = []
    for doc_id, meta, document, embedding in get_all_documents():
        if doc_id in exclude_ids:
            continue
        names = json.loads(meta["ingredient_names"])
        matched = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, n) for n in names)
        )
        if matched > 0:
            scored.append((matched, doc_id, meta, document, embedding))

    scored.sort(key=lambda x: -x[0])
    return scored[:max_candidates]


def retrieve_candidates_hybrid(
    user_text: str,
    user_ingredients: list,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )[0]

    # 1) kandidat dari SBERT (seperti sebelumnya)
    sbert_candidates = retrieve_candidates(user_text, top_k=top_k_sbert)
    existing_ids = {c["id"] for c in sbert_candidates}

    # 2) kandidat tambahan dari keyword search di seluruh koleksi
    kw_extra = keyword_candidates(
        user_ingredients, exclude_ids=existing_ids, max_candidates=max_keyword_extra
    )

    for matched, doc_id, meta, document, embedding in kw_extra:
        sbert_score = float(
            np.dot(query_embedding, np.array(embedding))
        )  # cosine, krn sudah dinormalisasi
        sbert_candidates.append(
            {
                "id": doc_id,
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "steps_text": meta["steps_text"],
                "sbert_score": sbert_score,
            }
        )

    return sbert_candidates


def rerank_hybrid(
    user_text: str,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
    top_n_final: int = 5,
    w_sbert: float = 0.5,
    w_efficiency: float = 0.5,
) -> tuple:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = (
            w_sbert * float(s_norm) + w_efficiency * eff["efficiency_score"]
        )

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], user_ingredients, len(candidates)

In [8]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

final_results, parsed_user_ingredients, n_pool = rerank_hybrid(
    user_input,
    top_k_sbert=30,
    max_keyword_extra=20,
    top_n_final=5,
    w_sbert=0.5,
    w_efficiency=0.5,
)

print("Bahan user (setelah parsing):", parsed_user_ingredients)
print("Total kandidat yang dibandingkan (SBERT + keyword fallback):", n_pool)
print("=" * 70)

for i, c in enumerate(final_results):
    print(f"[{i + 1}] {c['title']}")
    print(
        f"    final_score={c['final_score']:.4f}  "
        f"(sbert_norm={c['sbert_score_norm']:.3f}, efficiency={c['efficiency_score']:.3f})"
    )
    print(f"    bahan resep: {c['ingredient_names']}")
    print(f"    -> sudah dimiliki user: {c['matched_items']}")
    print(f"    -> perlu ditambah: {c['missing_items']}")
    print()

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
Total kandidat yang dibandingkan (SBERT + keyword fallback): 50
[1] Telur Dadar
    final_score=0.6518  (sbert_norm=0.859, efficiency=0.444)
    bahan resep: ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[2] Telur Dadar
    final_score=0.6320  (sbert_norm=0.864, efficiency=0.400)
    bahan resep: ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 'lada bubuk', 'garam']
    -> sudah dimiliki user: ['telur ayam', 'bawang putih bubuk']
    -> perlu ditambah: ['kornet sapi', 'bawang daun', 'keju cheddar', 'lada bubuk', 'garam']

[3] Telur Dadar Daun Bawang
    final_score=0.6250  (sbert_norm=1.000, efficiency=0.250)
    bahan resep: ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica b